### Source Material:
- Madi, Ramin. "NLP/Labs/Week_12/NMT." GitHub, ["https://github.com/raminmohammadi/NLP/tree/main/Labs/Week_12/NMT/Lab 1.ipynb"](https://github.com/raminmohammadi/NLP/blob/main/Labs/Week_12/NMT/Lab%201.ipynb)


# CoSQL Neural Machine translation using Seq2seq model

A **Seq2Seq** (Sequence-to-Sequence) model: neural network architecture used for tasks where the input and output are both sequences, but the length of the input sequence can be different from the output sequence. 


## Imports and Libraries

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, GRU, Dense
import numpy as np
from pathlib import Path

## Load Data
**Test Data Location:**
- 

In [3]:
# set input_texts as comma separted list of strings retrieved from train.txt where first pre-tab separated column is the input text and second column is the target text.

# initialize input_texts and target_texts as empty lists
input_texts_train = []
target_texts_train = []

# retrieve input_texts and target_texts from train.txt
train_path = Path(r"..\..\data\inputs\train.txt")
test_path = Path(r"..\..\data\inputs\test.txt")

with train_path.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 2:
            input_texts_train.append(parts[0])
            target_texts_train.append(parts[1].strip())

with test_path.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 2:
            # set input_texts as comma separated list of strings retrieved from train.txt where first pre-tab separated column is the input text and second column is the target text.

            # initialize input_texts and target_texts as empty lists
            input_texts_train = []
            target_texts_train = []
            input_texts_test = []
            target_texts_test = []

            # retrieve input_texts and target_texts from train.txt
            train_path = Path(r"..\..\data\inputs\train.txt")
            test_path = Path(r"..\..\data\inputs\test.txt")

            with train_path.open("r", encoding="utf-8") as f:
                for line in f:
                    parts = line.rstrip("\n").split("\t")
                    if len(parts) >= 2:
                        input_texts_train.append(parts[0])
                        target_texts_train.append(parts[1].strip())

            with test_path.open("r", encoding="utf-8") as f:
                for line in f:
                    parts = line.rstrip("\n").split("\t")
                    if len(parts) >= 2:
                        input_texts_test.append(parts[0])
                        target_texts_test.append(parts[1].strip())

            target_texts_test.append(parts[1].strip())

# print first 10 input_texts 
print("First 10 input_texts: ", input_texts_train[:10])

First 10 input_texts:  ["What's the grand total of points White has scored these playoffs?", 'How many points does Tatum average for Boston this postseason?', 'Overall, how many points has Jordan Walsh netted in the postseason?', 'How many games has Porzingis appeared in this postseason?', 'On average, how many points is Walsh putting up for Boston in the playoffs?', 'What is the number of playoff games Brown has played?', "What was Kornet's biggest scoring night of the playoffs for Boston?", 'How many playoff points has Queta accumulated in all?', "What is Payton Pritchard's lowest-scoring playoff game for the Celtics?", "What's the most points Springer has dropped in a playoff game for Boston?"]


## Tokenization

In [4]:
# Tokenize input and target texts
input_tokenizer = tf.keras.preprocessing.text.Tokenizer()
# Tokenize input and target texts
input_tokenizer = tf.keras.preprocessing.text.Tokenizer()
input_tokenizer.fit_on_texts(input_texts_train)
input_sequences = input_tokenizer.texts_to_sequences(input_texts_train)
input_sequences = tf.keras.preprocessing.sequence.pad_sequences(input_sequences, padding='post')

output_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
output_tokenizer.fit_on_texts(target_texts_train)
output_sequences = output_tokenizer.texts_to_sequences(target_texts_train)

## Add start and end tokens

In [5]:
# Step to add <start> and <end> tokens to the target sequences
def add_start_end_tokens_manually(tokenizer, sequences):
    if '<start>' not in tokenizer.word_index:
        tokenizer.word_index['<start>'] = len(tokenizer.word_index) + 1
    if '<end>' not in tokenizer.word_index:
        tokenizer.word_index['<end>'] = len(tokenizer.word_index) + 1

    updated_sequences = []
    for seq in sequences:
        updated_seq = [tokenizer.word_index['<start>']] + seq + [tokenizer.word_index['<end>']]
        updated_sequences.append(updated_seq)
    return tf.keras.preprocessing.sequence.pad_sequences(updated_sequences, padding='post')

In [6]:
# Add <start> and <end> tokens
output_sequences = add_start_end_tokens_manually(output_tokenizer, output_sequences)

In [7]:
# Vocabulary sizes
input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

## Model building

In [8]:

# Model parameters
embedding_dim = 256
units = 512
batch_size = 2


## Encoder model

The **Encoder** model: tensorflow keras. It consists of an **embedding layer** that converts input tokens into dense vectors, followed by a **Gated Recurrent Unit (GRU) layer**  that captures the temporal dependencies of the sequence. The model returns both the sequence of hidden states (useful for attention mechanisms) and the final hidden state (used as the context for the decoder). This encoded information helps the decoder generate the output sequence.

In [9]:
# Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.enc_units = enc_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = GRU(enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x):
        x = self.embedding(x)
        output, state = self.gru(x)
        return output, state

## Attention

In traditional Seq2Seq models, the encoder produces a single fixed-size context vector, which can sometimes fail to capture all the important information, especially in long sequences. To address this, **attention mechanisms** were introduced. Attention allows the decoder to focus on different parts of the input sequence at each decoding step, improving the model’s performance and handling long sequences better.

In [10]:
# Attention
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

## Decoder Model

The Decoder class defines the decoder part of a Seq2Seq model with attention. It receives the context vector from the encoder, the current hidden state, and the current token as input. 

It applies the Bahdanau attention mechanism to focus on important parts of the encoder's output, processes the input through an embedding layer and GRU, and produces a set of logits for the next token. 

It also returns the updated hidden state and attention weights, which are used in subsequent decoding steps and for interpreting which parts of the input sequence were most important at each timestep.

In [11]:
# Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.gru = GRU(dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = Dense(vocab_size)

        self.attention = BahdanauAttention(dec_units)

    def call(self, x, enc_output, hidden):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        x = self.fc(output)
        x = tf.squeeze(x, axis=1)
        return x, state, attention_weights

In [12]:
# Instantiate the models
encoder = Encoder(input_vocab_size, embedding_dim, units)
decoder = Decoder(output_vocab_size, embedding_dim, units)

## Model Training

In [13]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

In [14]:
def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

In [15]:
@tf.function
def train_step(input_seq, target_seq, enc_hidden):
    loss = 0

    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(input_seq)
        dec_hidden = enc_hidden

        dec_input = tf.expand_dims([output_tokenizer.word_index['<start>']] * batch_size, 1)

        for t in range(1, target_seq.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, enc_output, dec_hidden)
            loss += loss_function(target_seq[:, t], predictions)

            dec_input = tf.expand_dims(target_seq[:, t], 1)

    batch_loss = loss / int(target_seq.shape[1])
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss

In [16]:
# training call
EPOCHS = 10
for epoch in range(EPOCHS):
    enc_hidden = tf.zeros((batch_size, units))
    total_loss = 0

    for batch in range(len(input_sequences) // batch_size):
        batch_input = input_sequences[batch * batch_size:(batch + 1) * batch_size]
        batch_target = output_sequences[batch * batch_size:(batch + 1) * batch_size]

        batch_loss = train_step(batch_input, batch_target, enc_hidden)
        total_loss += batch_loss

    print(f'Epoch {epoch + 1}, Loss: {total_loss.numpy()}')


Epoch 1, Loss: 254.26060485839844
Epoch 2, Loss: 79.41548919677734
Epoch 3, Loss: 56.29523849487305
Epoch 4, Loss: 50.49198913574219
Epoch 5, Loss: 33.11894607543945
Epoch 6, Loss: 18.32572364807129
Epoch 7, Loss: 12.414586067199707
Epoch 8, Loss: 7.2083659172058105
Epoch 9, Loss: 5.092743396759033
Epoch 10, Loss: 3.4425787925720215


## translations

In [17]:
def translate(sentence):
    # Preprocess the input sentence
    sentence_seq = input_tokenizer.texts_to_sequences([sentence])
    sentence_seq = tf.keras.preprocessing.sequence.pad_sequences(sentence_seq, maxlen=input_sequences.shape[1], padding='post')

    enc_hidden = tf.zeros((1, units))
    enc_output, enc_hidden = encoder(sentence_seq)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([output_tokenizer.word_index['<start>']], 0)

    result = []

    for t in range(output_sequences.shape[1]):
        # Get the decoder's prediction
        predictions, dec_hidden, _ = decoder(dec_input, enc_output, dec_hidden)

        # Use greedy decoding: pick the token with the highest probability
        predicted_id = tf.argmax(predictions[0], axis=-1).numpy()

        # Get the predicted word from the tokenizer
        predicted_word = output_tokenizer.index_word.get(predicted_id, None)

        # If no valid predicted word, break the loop
        if predicted_word is None:
            #print(f"Unknown token ID {predicted_id} detected.")
            break

        # If we reach the <end> token, stop
        if predicted_word == '<end>':
            break

        # Append the predicted word to the result
        result.append(predicted_word)

        # Use the predicted word as the next input to the decoder
        dec_input = tf.expand_dims([predicted_id], 0)

    return ' '.join(result)


In [18]:

# overwrite the results file if it exists
results_file_path = Path("../../data/results/cosql_nmt_gru_seq2seq_model2_results.txt")
if results_file_path.exists():
    results_file_path.unlink()

#translate using the test.txt file and store in cosql_nmt_gru_seq2seq_model2_results.txt
with open(results_file_path, "w", encoding="utf-8") as f:
    for i in range(len(input_texts_test)):
        input_sentence = input_texts_test[i]
        target_sentence = target_texts_test[i]
        translated_sentence = translate(input_sentence)
        f.write(f"Translating sentence {i+1}: {input_sentence}\n")
        f.write(f"expected output: {target_sentence}\n")
        f.write(f"Translated: {translated_sentence}\n")
        f.write("\n")
        print(f"Translating sentence {i+1}: {input_sentence}")
        print(f"expected output: {target_sentence}")
        print(f"Translated: {translated_sentence}")
        print()

Translating sentence 1: Who put up the biggest point total in one playoff game?
expected output: SELECT PLAYER_NAME FROM boxscores ORDER BY PTS DESC LIMIT 1;
Translated: select player_name from boxscores order by pts desc limit 1;

Translating sentence 2: Name the player with the most points in any one playoff game.
expected output: SELECT PLAYER_NAME FROM boxscores ORDER BY PTS DESC LIMIT 1;
Translated: select player_name from boxscores where team_abbreviation = 'cle' order by pts desc limit 1;

Translating sentence 3: Which Boston player came up shortest on the scoreboard in a playoff game?
expected output: SELECT PLAYER_NAME FROM boxscores WHERE TEAM_ABBREVIATION = 'BOS' ORDER BY PTS ASC LIMIT 1;
Translated: select max(pts) from boxscores where player_name = ?;

Translating sentence 4: Name the Boston player with the lowest point total in a playoff game.
expected output: SELECT PLAYER_NAME FROM boxscores WHERE TEAM_ABBREVIATION = 'BOS' ORDER BY PTS ASC LIMIT 1;
Translated: select pl